In [ ]:
from unittest.mock import inplace

import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

from numpy.ma.extras import column_stack

# 设置随机种子
np.random.seed(42)
rows = 1000
# 预设一些随机池
countries = ['Argentina', 'Portugal', 'Brazil', 'Slovenia', 'Belgium', 'France', 'Germany', 'Spain', 'England']
clubs = ['FC Barcelona', 'Juventus', 'Paris Saint-Germain', 'Atletico Madrid', 'Real Madrid', 'Manchester City', 'Liverpool']
feet = ['Left', 'Right']
data = {
    'sofifa_id': [random.randint(150000, 250000) for _ in range(rows)],
    'player_url': [f"https://sofifa.com/player/{i}/" for i in range(rows)],
    'short_name': [f"Player_{i}" for i in range(rows)],
    'long_name': [f"Full Name of Player {i}" for i in range(rows)],
    'age': [random.randint(17, 38) for _ in range(rows)],
    'dob': [(datetime(2024, 1, 1) - timedelta(days=random.randint(18*365, 40*365))).strftime('%Y-%m-%d') for _ in range(rows)],
    'height_cm': [random.randint(165, 200) for _ in range(rows)],
    'weight_kg': [random.randint(60, 95) for _ in range(rows)],
    'nationality': [random.choice(countries) for _ in range(rows)],
    'club': [random.choice(clubs) for _ in range(rows)],
    'overall': [random.randint(50, 94) for _ in range(rows)],
    'value_eur': [random.randint(500000, 100000000) for _ in range(rows)],
    'wage_eur': [random.randint(1000, 500000) for _ in range(rows)],
    'preferred_foot': [random.choice(feet) for _ in range(rows)]
}
df = pd.DataFrame(data)
# 显示前 5 行，看看效果
print(df.head())

### **log() And iloc()**

In [ ]:
df.head(7)

In [ ]:
df.set_index('short_name',inplace=True)
df

In [ ]:
df[['age','dob','height_cm','weight_kg','nationality','club','overall']]

In [ ]:
df.loc[['Player_3']]

In [ ]:
df.loc['Player_3','age']

In [ ]:
df.loc[:,'age']

In [ ]:
df.loc[['Player_3','Player_89'],['height_cm','club']]

####  2. **Select a range**

In [ ]:
# start : stop : start
players=['Player_0','Player_9']

In [ ]:
df.loc[players,'age':'club']

In [ ]:
df.index

In [ ]:
df.loc['Player_0':'Player_9','age':'club']

#### 2.1  **Select with condition**

In [ ]:
df.loc[df['weight_kg']>80,'age':'club']

In [ ]:
df.loc[(df['weight_kg']>80) & (df['height_cm']>181),'age':'club']
# Multiple conditions

### **Above are loc , Now are iloc**
### **loc()选的是columns的名字，iloc选的是columns的index，可以把他看做坐标系**

In [ ]:
df.set_index('short_name',inplace=True)
df.iloc[23,8]
df.iloc[:,6]

In [ ]:
df.head()

In [ ]:
df.iloc[[1,7],2:6]
#左闭右开。2取6不取。0为sofifa_id，6为weight_kg，只取到5

In [ ]:
# With condition
list(df['weight_kg'>80])
# 语义：从 df 中选出那些 体重 > 80 的行，且显示 1 到 7 列 , Multiple Condition同样可用
df[df['weight_kg'] > 80].iloc[:, 1:7]

In [ ]:
df.loc['Player_0','height_cm']=190
df
#只有loc/iloc可以修改和定位数值！

In [ ]:
df.iloc[0,0]=114514
df

### 5 **Drop()**

In [ ]:
#axis=0删行，反之删列，不写默认为0，记得一定要写！
df.drop('Player_1',axis=0)
df.drop(index=['Player_4'])

In [ ]:
df.drop(['Player_13','Player_14'],axis=0,inplace=True)#记得想改就加个inplace
df

In [ ]:
df.drop('player_url',axis=1,inplace=True)
df

In [ ]:
df.columns[[6]]
df.drop(['long_name','age'],axis=1,inplace=True)
df

### 6 **Sample() and Random**

In [ ]:
df['dob'].sample(10,random_state=99)# 就是从哪里开始随机的意思

In [ ]:
df.sample(frac=0.3)
#sample方法就是随机取样，frac就是比率，抽出百分之多少的数据

In [ ]:
df.sample(frac=2,replace=True,random_state=100)
#Repalce是放回的意思，如果不放回那么就会报错，因为哪里来的2倍数据量呢

###  7. **Query() 可以按条件拿数据**

In [ ]:
re=df.query('age>30')#.sample(frac =  ...)
print(re)
#equals to df[df['age']>34]

In [ ]:
df.query("age > 34 and nationality == 'England'").head()

In [ ]:
df.query('not(weight_kg>90)').head()
#简单粗暴，和普通语法一样

In [ ]:
df.dtypes

#### **Astype () 可以强制转换字符类型**

In [ ]:
df['dob']=df['dob'].astype('datetime64[s]')
df['dob'].dt.year.head()

In [ ]:
df.query('dob.dt.year>1990').head()

#### 8. **Apply (),对row or column实现自定义函数的作用**

In [ ]:
df['age'].apply(np.sqrt).head()

In [ ]:
def bmi(row):
    return row['weight_kg']/(row['height_cm']/100)**2

In [ ]:
df.apply(bmi,axis=1)

#### **Lambda +apply()**

In [ ]:
# Lambda 表达式如下
sum_lambda = lambda a,b:a+b
sum_lambda(2,3)

In [ ]:
df['height_cm'].apply(lambda x:x/100).head(7)

In [ ]:
df['long_name'].apply(lambda x:x.upper())

In [ ]:
df['dob']=df['dob'].astype('datetime64[ns]')
df.dtypes

In [ ]:
df['dob'].apply(lambda x:x.year)

In [ ]:
df.apply(lambda x:x['weight_kg']/(x['height_cm'])**2,axis=1)

In [ ]:
df.apply(bmi,axis=1)

#### 9. **Copy ()**

In [ ]:
df_copy=df.copy(deep=False)
df_copy.loc['Player_3','height_cm']=190
df_copy
#这里的deep就是指深拷贝还是浅拷贝。deeo = True那么会修改到原来的表格那边去，相当于改地址了。